### Tools

Models can request to call tools that perform tasks such as fetching data from a database ,searching the web,or running code. Tools are pairings of:

1. A schema including the name of a tool,a description,and/or argument definitions(often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots Talk?")
response


AIMessage(content='The short answer is that parrots don’t actually "talk" in the human sense. They **mimic** sounds. They are not using language to communicate ideas, feelings, or questions the way humans do. Instead, they are excellent vocal mimics.\n\nHere’s why they do it:\n\n### 1. **Social Bonding**\nIn the wild, parrots are highly social flock animals. They communicate constantly with calls, whistles, and chatter to:\n- Keep in touch with their flock.\n- Coordinate group movements.\n- Strengthen social bonds.\n\nWhen a parrot is in captivity, **you become its flock**. By mimicking the sounds of their human companions, parrots are trying to bond and stay connected with their "group."\n\n### 2. **Vocal Ability**\nParrots have a unique vocal anatomy:\n- They have a syrinx (vocal organ) located at the base of the trachea, which allows them to produce a wide range of sounds.\n- Their beaks and tongues are highly flexible, enabling them to shape sounds with surprising precision.\n- Thi

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])    


In [4]:
response = model_with_tools.invoke("What's the weather in Siliguri?")
print(response)
for tool_call in response.tool_calls:
    #View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'tool_calls': [{'id': 'ddedd7bmz', 'function': {'arguments': '{"location":"Siliguri"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 279, 'total_tokens': 307, 'completion_time': 0.076801753, 'completion_tokens_details': None, 'prompt_time': 0.019605214, 'prompt_tokens_details': None, 'queue_time': 0.068041046, 'total_time': 0.096406967}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0a905-9bc3-7202-bdd6-3adc2f035b6e-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Siliguri'}, 'id': 'ddedd7bmz', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 279, 'output_tokens': 28, 'total_tokens': 307}
Tool: get_weather
Args: {'location': 'Siliguri'}


#### Tool Execution Loops


In [5]:
# Step 1: Model generates tool calls
messages =  [{"role": "user", "content": "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute Tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in NewYork is Sunny"



It's sunny in New York! 🌞


In [6]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'zbgv1k0sx', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 279, 'total_tokens': 306, 'completion_time': 0.07294647, 'completion_tokens_details': None, 'prompt_time': 0.019533657, 'prompt_tokens_details': None, 'queue_time': 0.051051442, 'total_time': 0.092480127}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a90f-0f2c-7722-9af9-9a70ad7111d0-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'zbgv1k0sx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 279, 'output_tokens': 27, 'total_tokens': 306}),
 ToolMessage(content="